# Блок 3: SQL

---

## Задание 1. Абитуриенты

### Условие

Таблица `examination`:
- `id` — ID абитуриента
- `scores` — количество баллов дополнительного вступительного испытания (0–100)

Требуется реализовать запрос, который создаёт колонку с позицией абитуриента в общем рейтинге.

### Решение

Используется оконная функция `RANK()`. Абитуриенты с одинаковыми баллами получают одинаковую позицию, следующая позиция при этом пропускается — что соответствует стандартной логике рейтинговых списков.

```sql
SELECT
    id,
    scores,
    RANK() OVER (ORDER BY scores DESC) AS position
FROM examination;
```

### Пример результата

| id | scores | position |
|----|--------|----------|
| 7  | 98     | 1        |
| 3  | 95     | 2        |
| 12 | 95     | 2        |
| 1  | 87     | 4        |
| 5  | 72     | 5        |

Два абитуриента с одинаковыми баллами (95) занимают позицию 2. Следующий абитуриент получает позицию 4, а не 3.

---

## Задание 2. FULL JOIN

### Условие

Первая таблица содержит 30 строк, вторая — 20 строк. Выполняется операция `FULL JOIN`. Ключи для соединения могут быть как полностью совпадающими, так и абсолютно уникальными.

Требуется определить диапазон возможного количества строк в результирующей таблице.

### Решение

`FULL JOIN` возвращает все строки из обеих таблиц. Совпавшие по ключу строки объединяются в одну строку, несовпавшие дополняются `NULL` с противоположной стороны.

**Минимум** достигается, когда все 20 ключей второй таблицы совпадают с ключами первой:
- 20 совпавших пар + 10 строк из первой таблицы без пары = **30 строк**

**Максимум** достигается, когда ни один ключ не совпадает:
- 30 строк из первой таблицы + 20 строк из второй = **50 строк**

### Ответ

**Минимально 30 и максимально 50 строк.**

---

## Задание 3. Покупки

### Условие

```sql
CREATE TABLE account
(
    id        integer,
    client_id integer,
    open_dt   date,
    close_dt  date
);

CREATE TABLE transaction
(
    id               integer,
    account_id       integer,
    transaction_date date,
    amount           numeric(10,2),
    type             varchar(3)
);
```

Вывести ID клиентов, которые за последний месяц по всем своим счетам совершили покупок менее чем на 5000 рублей. Без использования подзапросов и оконных функций.

### Решение

Используется `LEFT JOIN`, чтобы учесть в том числе клиентов, у которых за период не было ни одной покупки — их суммарная сумма равна 0, что удовлетворяет условию `< 5000`.

Условия фильтрации по периоду и типу транзакции вынесены в `ON`, а не в `WHERE` — иначе `LEFT JOIN` превратится во внутренний и клиенты без транзакций будут исключены из результата.

```sql
SELECT
    a.client_id
FROM account a
LEFT JOIN transaction t
    ON  t.account_id = a.id
    AND t.type = 'buy'
    AND t.transaction_date >= DATE_TRUNC('month', CURRENT_DATE - INTERVAL '1 month')
    AND t.transaction_date <  DATE_TRUNC('month', CURRENT_DATE)
GROUP BY
    a.client_id
HAVING
    COALESCE(SUM(t.amount), 0) < 5000;
```

### Пояснения к запросу

| Элемент | Назначение |
|---------|------------|
| `LEFT JOIN` | Включает клиентов без покупок в указанном периоде |
| `t.type = 'buy'` | Фильтрация только по покупкам |
| `DATE_TRUNC('month', CURRENT_DATE - INTERVAL '1 month')` | Начало прошлого календарного месяца |
| `DATE_TRUNC('month', CURRENT_DATE)` | Начало текущего месяца (граница периода) |
| `GROUP BY a.client_id` | Агрегация по клиенту — суммирует покупки по всем его счетам |
| `COALESCE(SUM(t.amount), 0)` | Заменяет NULL нулём для клиентов без транзакций |
| `HAVING ... < 5000` | Оставляет только клиентов с суммой покупок менее 5000 руб. |

Подзапросы и оконные функции не использованы.

---

## Итоговая сводка

| Задание | Ключевые инструменты | Результат |
|---------|----------------------|-----------|
| 1. Абитуриенты | `RANK() OVER (ORDER BY scores DESC)` | Позиция в рейтинге для каждого абитуриента |
| 2. FULL JOIN | Теория JOIN-операций | Минимум 30, максимум 50 строк |
| 3. Покупки | `LEFT JOIN`, `GROUP BY`, `HAVING`, `COALESCE` | ID клиентов с суммой покупок < 5000 руб. за последний месяц |

---

*Запросы написаны для PostgreSQL.*